# 02 Feature checks
Run `scripts/build_features.py` first.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from ytclickbait.config import load_config

cfg = load_config()
videos = pd.read_parquet(cfg.file("videos"))
split = pd.read_parquet(cfg.file("split"))
sns.set_theme(style="whitegrid")
from ytclickbait.features.tfidf import load_tfidf, top_terms

rules = pd.read_parquet(cfg.file("rule_features"))
sent = pd.read_parquet(cfg.file("sentiment"))
df = videos.merge(rules, on="video_id").merge(sent, on="video_id")

## Rule features

In [ ]:
rules.drop(columns="video_id").describe().T

In [ ]:
feature_cols = [c for c in rules.columns if c != "video_id"]
corr = df[feature_cols + ["log_views"]].corr()["log_views"].drop("log_views").sort_values()
corr.plot.barh(figsize=(6, 6))
plt.title("Raw correlation with log(views), no controls")

## Clickbait features by category

In [ ]:
cols = ["caps_ratio", "n_exclaim", "n_clickbait_words", "n_emoji"]
df.groupby("category_name")[cols].mean().sort_values("n_clickbait_words")

## Sentiment
Do VADER and RoBERTa agree?

In [ ]:
if "roberta_score" in sent:
    sample = sent.sample(min(5000, len(sent)), random_state=0)
    sns.scatterplot(data=sample, x="vader_compound", y="roberta_score", alpha=0.2)
    print("Correlation:", sent["vader_compound"].corr(sent["roberta_score"]).round(3))

## TF-IDF

In [ ]:
vectorizer, X = load_tfidf(cfg.file("tfidf_matrix"), cfg.file("tfidf_vectorizer", "models"))
print(X.shape)
top_terms(vectorizer, X, 40)

## Embeddings
Nearest neighbours of a sample title, as a sanity check.

In [ ]:
emb = np.load(cfg.file("embeddings"))
i = 0
sims = emb @ emb[i]
print("Query:", videos.loc[i, "title"])
videos.loc[np.argsort(-sims)[1:11], ["title", "category_name"]]